# Preview a web app while your coding agent builds it

A coding agent runs in an Amazon Bedrock AgentCore Runtime session and keeps a Vite website running beside it. In this notebook you open one local link: a chat with the agent on the left, the live website on the right. Ask for a change, watch the agent's steps arrive, and see the website update as each file is saved.

![The agent preview page: chat on the left, the live site on the right](images/ui-after.png)

Files: [`agent/app.ts`](agent/app.ts) the agent, [`site/`](site) the website it edits, [`preview_app.py`](preview_app.py) and [`ui.html`](ui.html) the local page, [`deploy.py`](deploy.py), [`cleanup.py`](cleanup.py).

## Prerequisites

- An AWS account in a Region where AgentCore Runtime Session endpoints are available, with credentials configured, and permission to create an IAM role, upload to Amazon S3, create a runtime, and call `bedrock-agentcore:InvokeAgentRuntime` and `bedrock-agentcore:InvokeAgentRuntimeProxy` on it.
- Amazon Bedrock model access for Anthropic Claude Haiku 5.5.
- Python 3.10 or later with bedrock-agentcore 1.26.0 or later, and Node.js 22 or later with `npm`, which `deploy.py` uses to package the agent and the website.

In [ ]:
%pip install -q -r requirements.txt

## 1. Deploy the agent and its website

`deploy.py` packages the agent and the website into one zip, uploads it to Amazon S3, and creates the runtime on the managed `NODE_22` runtime, so you need no container image.

The runtime allows two ports in `allowedPorts`, which are the ports the Session endpoint can reach: 8090, where the website's dev server listens, and 8081, where the agent reports what it changed. This takes a few minutes.

In [ ]:
%run deploy.py

## 2. Open the preview

`PreviewApp` does three things. It starts a session, which starts the website. It opens a local endpoint with the SDK's Session endpoint client, which adds your AWS credentials to every request your browser makes and relays the WebSocket the website uses for live updates. And it serves the page on `localhost:8087`.

Open the link and click one of the suggested requests, for example *Switch this website to a dark coffee-house color theme*. The agent's steps appear in the chat, and when it saves a file the website on the right changes without a refresh, with a green banner naming the file it came from. That change was made inside the session and reached your browser over the dev server's WebSocket, through the Session endpoint. If you used Run All, the notebook waits at the clean up step until you press Enter, so the preview stays up while you use it.

In [ ]:
from preview_app import PreviewApp

app = PreviewApp().start()
print("Open", app.url)

## 3. Or ask from Python

The page sends each prompt to `InvokeAgentRuntime`, and the agent streams its steps back. You can do the same from here. The website in your browser updates either way.

In [ ]:
for event in app.ask("Make the Order ahead button green"):
    if "step" in event:
        print("  ", event["step"])
    elif "done" in event:
        print(event["done"]["reply"])

## What the page used

The page never sees the Session endpoint URL or a credential. Here is what it used. The Session endpoint URL came back in the `InvokeAgentRuntime` response, and the local endpoint reaches port 8090 by putting the port in front of the hostname. The same URL reaches the agent's own port, 8081, where it lists each prompt and the files it changed.

A port outside `allowedPorts` is refused, and so is a request without credentials, which is why the browser goes through the local endpoint.

In [ ]:
import urllib.error
import urllib.request

from bedrock_agentcore.runtime import SessionEndpointResponse

status = app.endpoint.request("GET", "/status", port=8081)
print("GET /status on port 8081: HTTP", status.status)
for edit in status.json()["edits"]:
    print("  ", edit["prompt"], "->", ", ".join(edit["files"]))

refused = app.endpoint.request("GET", "/", port=9999)
print("GET / on port 9999: HTTP", refused.status, refused.proxy_error)

try:
    urllib.request.urlopen(app.endpoint.url.replace("https://", "https://8090--", 1) + "/", timeout=30)
except urllib.error.HTTPError as error:
    code = SessionEndpointResponse(status=error.code, headers=dict(error.headers)).proxy_error
    print("unsigned GET / on port 8090: HTTP", error.code, code)

## Clean up

Stop the page, the local endpoint, and the session, then remove the runtime, the uploaded code, and the IAM role. The website the agent edited lived in the session, so it goes with the session.

In [ ]:
input("Press Enter when you are done with the preview, to stop it and delete the runtime ")
app.stop()
print("preview and session stopped")

In [ ]:
%run cleanup.py